**TF** _- сколько раз встречается в конкретной ячейке. Значение слова для отдельной ячейки._


**IDF** _- сколько раз встречается во всех ячейках, (кол-во ячеек) / (кол-во встреч во всех ячейках). Значение слова для всего документа._

_В формуле IDF присутствует **логарифм**, чтобы разница между почти близкими (100 ячеек/в 10 документах встретили слово) и (100 ячеек/в 1 документе встретили слово) не была слишком большой._

### **Возьмём текстовый признак.**

In [14]:
import pandas as pd

x = pd.read_csv('8 ks_crashed.csv')['Название']
#удалим пропущенные значения
x.dropna()
#также возьмём не все строки, чтобы было быстрее
x = x.iloc[330_000:]
#посмотрим на столбик
x.head()

330000                      Brew & Beast, Bacon & Beer Kits
330001    New comic 'Arthur Shilling' Issue #1 Vs The Ne...
330002                               Dream Traveler: Africa
330003                                         Hydro-Anthem
330004    William Kentridge creates art history on RomeÔ...
Name: Название, dtype: str

_Мы хотим обработать эту колонку так, чтобы мы узнали TF умноженное на IDF для каждого из слов. Тем самым для каждого объекта мы получим вектор._

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

#возьмём объект
tfidf = TfidfVectorizer()
tfidf.fit(x)
#хранит данные нужные для ITF IDF

array([7.7310181 , 7.32555299, 7.7310181 , ..., 7.03787092, 7.03787092,
       7.7310181 ], shape=(4066,))

_Посмотрим на первый объект выборки._

In [15]:
first = x[330000]
first

'Brew & Beast, Bacon & Beer Kits'

_Превратим его в TF-IDF-вектор, используя словарь и IDF, которые мы узнали при fit(x)_

In [18]:
#векторизуем текст, то есть узнаем TF * IDF
tfidf.transform([first])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 4066)>

_Мы получили разряженную матрицу - матрицу с очень большим кол-вом нулей. Превратим её в обычную._

In [19]:
tfidf.transform([first]).todense()

matrix([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 4066))

_Посмотрим на содержимое вектора._

In [20]:
df = pd.DataFrame(
tfidf.transform([first]).T.todense(),
index=tfidf.get_feature_names_out(),
columns=['tfidf'])

df.sort_values('tfidf', ascending=False)

,tfidf
beer,0.465019
beast,0.465019
bacon,0.440630
kits,0.440630
brew,0.423326
...,...
1977,0.000000
1995,0.000000
19xx,0.000000
1st,0.000000


_Узнаем информацию только о первой ячейке в наших объектах._

In [21]:
df[df.values > 0]

,tfidf
bacon,0.440630
beast,0.465019
beer,0.465019
brew,0.423326
kits,0.440630
